# Lab 09 — Fit a Tiny Scaling Law
**Goal:** Fit a simple power-law relationship to synthetic validation-loss measurements and use it to extrapolate carefully.

Use the loop: predict → run → inspect → break → decide.


In [ ]:
import numpy as np
np.random.seed(0)
N=np.array([20,50,100,200,500,1000])*1e6
true_a,true_b,true_c=2.0,0.08,1.65
loss=true_a+true_b*N**(-0.2)+0.02*np.random.randn(len(N))
print(list(zip((N/1e6).astype(int),loss.round(4))))


## Prediction
Predict whether a power law fit can safely tell you the loss at a model 100x larger than the largest training run.


In [ ]:
from scipy.optimize import curve_fit
def law(n,a,b,alpha): return a+b*n**(-alpha)
p,_=curve_fit(law,N,loss,p0=[2,1,0.2],maxfev=10000)
print("fit:",p)
for n in [2e3,5e3,1e4]:
    print("extrapolated",n/1e6,"M params ->",law(n,*p))


## Interpretation
Scaling laws can be useful interpolation/extrapolation tools, but extrapolation risk grows when the regime changes or the fitted data are too narrow.


In [ ]:
# Break the model by removing the largest points and refitting.
mask=N<=200e6
p2,_=curve_fit(law,N[mask],loss[mask],p0=[2,1,0.2],maxfev=10000)
print("fit with less evidence:",p2)
print("large-model extrapolation:",law(1e10,*p2))


## Answers
Treat scaling-law coefficients as empirical estimates under a defined experimental regime. Use multiple runs, hold important variables controlled, and report uncertainty rather than presenting an extrapolation as a fact.

### Primary source
https://arxiv.org/abs/2203.15556


## Deepening: inspect extrapolation risk

This section turns the notebook into a quantitative lab. Predict before executing.


In [ ]:
import numpy as np

compute = np.array([1, 2, 4, 8, 16], dtype=float)
loss = np.array([2.5, 2.05, 1.72, 1.48, 1.31], dtype=float)

x = np.log(compute)
y = np.log(loss)
slope, intercept = np.polyfit(x, y, 1)
print("log-log slope:", slope)
print("fit implies loss proportional to compute^", slope)

for c in [32, 64, 128]:
    pred = np.exp(intercept) * c**slope
    print("extrapolated", c, "->", pred)


## Break experiment

Fit using only the first three compute points and extrapolate to 16 or 32.

Compare the prediction with observed points.

### Report

observed regime → fitted parameters → extrapolation range → prediction error → why the model may fail outside the data range

Complete templates/EXPERIMENT_CARD.md for the final result.


## 7. Research exercise — compute budget from real corpus scale


In [ ]:
from datasets import load_dataset

sources=[
    ("HuggingFaceFW/fineweb","sample-10BT"),
    ("HuggingFaceFW/fineweb-edu","sample-10BT"),
]
for ds_id,config in sources:
    ds=load_dataset(ds_id,config,split="train",streaming=True)
    seen=0
    token_sum=0
    for row in ds:
        n=row.get("token_count")
        if n:
            token_sum+=int(n)
            seen+=1
        if seen>=500: break
    print(ds_id, "sampled_docs=",seen,"sampled_tokens=",token_sum)

# For a fixed dense-model compute budget, N*D is approximately constant.
C=1e21
for N in [0.1e9,0.3e9,1e9]:
    D=C/(6*N)
    print(f"N={N/1e9:.1f}B -> D≈{D/1e9:.1f}B tokens under 6ND heuristic")


<details><summary><strong>Expected findings / answer</strong></summary>

The real dataset determines what token supply is available, but it does not by itself determine the compute-optimal model size. Under the first-order $6ND$ heuristic, increasing parameter count at fixed compute forces the training-token budget down. The research decision is therefore to fit the scaling relationship only inside the measured regime and then test a new point; do not treat a theoretical allocation as a validated optimum.
</details>
